## Make src/ importable and load project modules

In [1]:
# The notebook and code are in different directories inside the main project dir.
# Hence, we have to inform python where to find it.
# We add the project's src/ folder to the import path.
import sys
from pathlib import Path

SRC = Path.cwd().parent / "src"
sys.path.insert(0, str(SRC))

# Import project modules
import config
import cleaning

# Import other external libraries
import pandas as pd

print("Imports OK. Project root:", config.PROJECT_ROOT)

Imports OK. Project root: /home/koala/lab/adversec


## Load the raw data

In [2]:
# Call the loader.
# It walks the six files in config.RAW_FILES, tags each row with its true class, and stacks them into a DF.
raw = cleaning.load_raw_data()

# Show the overall shape
print("\nCombined shape:", raw.shape)

# Peak at the first few raws
raw.head()

 loaded decimal_benign.csv                         rows=1,223,737
 loaded decimal_DoS.csv                            rows=   74,663
 loaded decimal_spoofing-GAS.csv                   rows=    9,991
 loaded decimal_spoofing-RPM.csv                   rows=   54,900
 loaded decimal_spoofing-SPEED.csv                 rows=   24,951
 loaded decimal_spoofing-STEERING_WHEEL.csv        rows=   19,977

Combined shape: (1408219, 13)


,ID,DATA_0,DATA_1,DATA_2,DATA_3,DATA_4,DATA_5,DATA_6,DATA_7,label,category,specific_class,true_class
0,65,96,0,0,0,0,0,0,0,BENIGN,BENIGN,BENIGN,benign
1,1068,132,13,160,0,0,0,0,0,BENIGN,BENIGN,BENIGN,benign
2,535,127,255,127,255,127,255,127,255,BENIGN,BENIGN,BENIGN,benign
3,131,15,224,0,0,0,0,0,0,BENIGN,BENIGN,BENIGN,benign
4,936,1,0,39,16,0,0,0,0,BENIGN,BENIGN,BENIGN,benign


## Audit the raw duplication

In [4]:
# The signature columns: the 9 CAN features plus the true class.
# Two rows are "the same" only if all nine values AND the class match.
signature_cols = config.FEATURE_COLUMNS + ["true_class"]

# Run the audit over the raw data
raw_audit = cleaning.audit_duplication(raw, subset=signature_cols)

# Print the findings
print("RAW DUPLICATION AUDIT")
print(f"    total rows          : {raw_audit['total_rows']:>10,}")
print(f"    unique signatures   : {raw_audit['unique_signatures']:>10,}")
print(f"    duplicate rows      : {raw_audit['duplicate_rows']:>10,}")
print(f"    duplication rate    : {raw_audit['duplication_rate_pct']:.4f}%")

RAW DUPLICATION AUDIT
    total rows          :  1,408,219
    unique signatures   :      3,588
    duplicate rows      :  1,404,631
    duplication rate    : 99.7452%


## Strict de-duplication

In [6]:
# Run strict de-dup: keep one row per unique (features + class) signature
strict = cleaning.strict_dedup(raw, config.FEATURE_COLUMNS)

print("STRICT DE-DUPLICATION")
print(f"    rows before     : {len(raw):>10,}")
print(f"    rows after      : {len(strict):>10,}")
print(f"    kept            : {len(strict) / len(raw) * 100:.4f}% of original\n")

# The critical view: how many unique signatures per class.
# value_counts() tallies each class; we sort by index so the classes read in a stable order.
print("Unique signatures per class")
print(strict["true_class"].value_counts().sort_index())

STRICT DE-DUPLICATION
    rows before     :  1,408,219
    rows after      :      3,588
    kept            : 0.2548% of original

Unique signatures per class
true_class
DoS                          21
benign                     3547
spoofing-GAS                  2
spoofing-RPM                 10
spoofing-SPEED                5
spoofing-STEERING_WHEEL       3
Name: count, dtype: int64


## Stage 1 Summary: The Accuracy Trap, Confirmed

This notebook loads, audits, and de-duplicates the CICIoV2024 decimal dataset.

### What we found

| Metric | Value |
|---|---|
| Total raw rows | 1,408,219 |
| Unique signatures (features + class) | **3,588** |
| Duplicate rows | 1,404,631 |
| Duplication rate | **99.75%** |

The raw dataset is 99.75% redundant copies. This independently reproduces the
finding of Le & Alsmadi (2026) on our own copy of the data, and is the empirical
basis for the *accuracy trap*: the near-perfect F1 scores reported in prior work
are earned on data where each pattern is repeated tens of thousands of times.

### Unique signatures per class (strict de-duplication)

| Class | Raw rows | Unique signatures |
|---|---|---|
| benign | 1,223,737 | 3,547 |
| DoS | 74,663 | 21 |
| spoofing-RPM | 54,900 | 10 |
| spoofing-SPEED | 24,951 | 5 |
| spoofing-STEERING_WHEEL | 19,977 | 3 |
| spoofing-GAS | 9,991 | 2 |
| **Total** | **1,408,219** | **3,588** |

Only **41 unique attack signatures** exist across all five attack types. The GAS
attack is just 2 distinct frames repeated roughly 10,000 times. A model scoring a
perfect F1 here has memorised a handful of patterns, not learned to detect
intrusions.

### Implication for the experiment

The strict set cannot be trained honestly as a six-class problem: a class of 2–5
examples is not learnable. Naive duplication (repeating those few frames) would
only recreate the accuracy trap. The augmentation strategy is therefore:

- **Strict set** → truth/analysis set; exposes the trap.
- **Light duplication** → solves convergence (gives the optimiser enough volume
  to train a baseline and form gradients).
- **Adversarial synthesis (FGSM/PGD)** → solves diversity (generates new
  near-miss samples around the real signatures).

Adversarial samples will be generated from **training-fold signatures only**,
with the test set held as **real signatures**, to prevent the trap returning in
disguise through train/test leakage.